### University of Virginia
### DS 5110: Big Data Systems

### Lab: Supervised Learning
### Last updated: March 2, 2023

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [1]:
from pyspark.sql import SparkSession

DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 11:18:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
sc = spark.sparkContext

#### Enter code and solution

In [3]:
df1 = spark.read.csv(DATA_FILEPATH, inferSchema=True, header = True)
df1.show(2)

26/10/08 11:18:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+---------+-----+-----+-----+------+-------+-------+------+-------+------+-------+------+------+-----+-----+--------+-------+-------+-------+-------+--------+-----+-----+-----+------+------+------+------+------+------+-------+
|    id|diagnosis|   f1|   f2|   f3|    f4|     f5|     f6|    f7|     f8|    f9|    f10|   f11|   f12|  f13|  f14|     f15|    f16|    f17|    f18|    f19|     f20|  f21|  f22|  f23|   f24|   f25|   f26|   f27|   f28|   f29|    f30|
+------+---------+-----+-----+-----+------+-------+-------+------+-------+------+-------+------+------+-----+-----+--------+-------+-------+-------+-------+--------+-----+-----+-----+------+------+------+------+------+------+-------+
|842302|        M|17.99|10.38|122.8|1001.0| 0.1184| 0.2776|0.3001| 0.1471|0.2419|0.07871| 1.095|0.9053|8.589|153.4|0.006399|0.04904|0.05373|0.01587|0.03003|0.006193|25.38|17.33|184.6|2019.0|0.1622|0.6656|0.7119|0.2654|0.4601| 0.1189|
|842517|        M|20.57|17.77|132.9|1326.0|0.08474|0.07864|0.086

In [4]:
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

df1 = df1.withColumn("label", F.when(F.col("diagnosis") == "M", 1.0).otherwise(0.0))

assembler = VectorAssembler(inputCols=["f1", "f2"],
                            outputCol="features")

df1 = assembler.transform(df1)
df1.select("*").show(2, truncate=False)

+------+---------+-----+-----+-----+------+-------+-------+------+-------+------+-------+------+------+-----+-----+--------+-------+-------+-------+-------+--------+-----+-----+-----+------+------+------+------+------+------+-------+-----+-------------+
|id    |diagnosis|f1   |f2   |f3   |f4    |f5     |f6     |f7    |f8     |f9    |f10    |f11   |f12   |f13  |f14  |f15     |f16    |f17    |f18    |f19    |f20     |f21  |f22  |f23  |f24   |f25   |f26   |f27   |f28   |f29   |f30    |label|features     |
+------+---------+-----+-----+-----+------+-------+-------+------+-------+------+-------+------+------+-----+-----+--------+-------+-------+-------+-------+--------+-----+-----+-----+------+------+------+------+------+------+-------+-----+-------------+
|842302|M        |17.99|10.38|122.8|1001.0|0.1184 |0.2776 |0.3001|0.1471 |0.2419|0.07871|1.095 |0.9053|8.589|153.4|0.006399|0.04904|0.05373|0.01587|0.03003|0.006193|25.38|17.33|184.6|2019.0|0.1622|0.6656|0.7119|0.2654|0.4601|0.1189 |1.0  

In [5]:
splits = df1.randomSplit([0.6, 0.4], 314)
train = splits[0]
test = splits[1]

scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", 
                       withMean=True, withStd=True)
scalerModel = scaler.fit(train)
train_scale = scalerModel.transform(train) 
test_scale = scalerModel.transform(test)

In [6]:
lr = LogisticRegression(labelCol='label',
                        featuresCol='scaledFeatures',
                        fitIntercept = True)

lrModel = lr.fit(train_scale)

print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))

26/10/08 11:18:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Coefficients: [3.5700446280343403,1.0818379381825267]
Intercept: -0.45507294884565347


In [7]:
lrPred = lrModel.transform(test_scale)

evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction",
                                          labelCol="label",
                                          metricName="areaUnderROC")

auc = evaluator.evaluate(lrPred)

print("Test set Area Under ROC:", auc)

Test set Area Under ROC: 0.9514088556641738


#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [8]:
import os
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [9]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

In [10]:

from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

#### Enter code and solution

In [11]:
df2 = spark.read.csv(DATA_FILEPATH2,  inferSchema=True, header = True)

df2 = df2.withColumn("median_house_value", F.col("median_house_value") / 100000)

df2 = df2.withColumn("rooms_per_household", F.col("total_rooms") / F.col("households"))

In [12]:
splits = df2.randomSplit([0.8, 0.2], 314)
train = splits[0]
test = splits[1]

feats = ["total_bedrooms", "population", "households", "median_income", "rooms_per_household"]

assembler = VectorAssembler(inputCols=feats,
                            outputCol="features")

train_assembler = assembler.transform(train)
test_assembler = assembler.transform(test)
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", 
                       withMean=True, withStd=True)

scalerModel = scaler.fit(train_assembler) 
train_scale = scalerModel.transform(train_assembler) 
test_scale = scalerModel.transform(test_assembler)

In [13]:

lr = LinearRegression(featuresCol='scaledFeatures',         
                      labelCol='median_house_value',
                      maxIter=10,
                      regParam=0.3, 
                      elasticNetParam=0.8)

lrModel = lr.fit(train_scale)

print("Coefficients:", lrModel.coefficients)
print("Intercept:", lrModel.intercept)

Coefficients: [0.0,0.0,0.0,0.5228380318201065,0.0]
Intercept: 2.070762241986439


In [14]:
lrPred = lrModel.transform(test_scale)
ev = RegressionEvaluator(predictionCol="prediction", labelCol="median_house_value")

print('-'*20)
print("METRICS")
print("Mean Squared Error:", ev.evaluate(lrPred, {ev.metricName: "mse"}))
print("R Squared:", ev.evaluate(lrPred, {ev.metricName:'r2'}))

--------------------
METRICS
Mean Squared Error: 0.7551749809614869
R Squared: 0.434727690087484
